# CP1 · Notebook 04 — Comparativa Clásico vs Deep Learning

## Objetivo

Producir la **tabla y el grid visual** que demuestran dónde gana cada enfoque. Este notebook **no re-infiere**: carga los resultados que 02 y 03 dejaron en `outputs/` y los compara.

**Salida esperada**:
1. Grid visual 3×3 (3 categorías × 3 imágenes) con `original | clásico | DL` lado a lado.
2. Tabla cuantitativa: tasa de detección × categoría × método.
3. Gráfica de tiempos (boxplot clásico vs DL).
4. JSON de resumen final.

**Tiempo estimado**: 10 min.
**Requiere**: `02_clasico_canny_hough.ipynb` y `03_deep_learning.ipynb` ejecutados con éxito (sus JSON en `outputs/` y los overlays DL en `outputs/03_dl_overlays/`).

## 0. Imports y verificación de prerequisitos

In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATASET_DIR = Path('../datasets/lanes-subset')
OUT_DIR     = Path('../outputs')

needed = {
    'clásico (resumen)':  OUT_DIR / '02_classical_summary.json',
    'DL (resumen)':       OUT_DIR / '03_dl_summary.json',
    'DL (overlays)':      OUT_DIR / '03_dl_overlays',
}
missing = [name for name, p in needed.items() if not p.exists()]
if missing:
    raise SystemExit(
        'Faltan outputs de notebooks anteriores: '
        f"{', '.join(missing)}.\n"
        'Ejecuta 02 y 03 antes de seguir.'
    )

with open(needed['clásico (resumen)']) as f:
    classical = json.load(f)
with open(needed['DL (resumen)']) as f:
    dl = json.load(f)

print('✅ resúmenes cargados')
print(f"   clásico: {classical['total_images']} imágenes, {classical['mean_time_ms']:.1f} ms/img medio")
print(f"   DL:      {dl['total_images']} imágenes, {dl['mean_time_ms']:.1f} ms/img medio")

## 1. Re-ejecutar el clásico sobre las 50 imágenes (para tener overlays)

El JSON del clásico (notebook 02) tiene métricas pero no los overlays. Es muy rápido (<30 s en CPU) regenerarlos aquí. Reusamos la función `detect_lanes_classical` definiéndola compactamente.

In [ ]:
import time

def make_roi_trapezoid(image_shape):
    h, w = image_shape[:2]
    polygon = np.array([
        [(int(0.05 * w), h),
         (int(0.45 * w), int(0.60 * h)),
         (int(0.55 * w), int(0.60 * h)),
         (int(0.95 * w), h)]
    ], dtype=np.int32)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, polygon, 255)
    return mask, polygon[0]

def average_lane(lines, side, image_shape, min_slope=0.4):
    if lines is None: return None
    h = image_shape[0]
    slopes, intercepts = [], []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        if x2 == x1: continue
        slope = (y2 - y1) / (x2 - x1)
        if abs(slope) < min_slope: continue
        if side == 'left' and slope > 0: continue
        if side == 'right' and slope < 0: continue
        slopes.append(slope)
        intercepts.append(y1 - slope * x1)
    if not slopes: return None
    slope, intercept = float(np.mean(slopes)), float(np.mean(intercepts))
    y1, y2 = h, int(0.6 * h)
    return (int((y1 - intercept) / slope), y1, int((y2 - intercept) / slope), y2)

def detect_lanes_classical(image, canny_low=50, canny_high=150, blur_kernel=5,
                            hough_threshold=40, hough_min_length=30, hough_max_gap=100):
    t0 = time.perf_counter()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (blur_kernel, blur_kernel), 0)
    edges = cv2.Canny(blurred, canny_low, canny_high)
    mask, _ = make_roi_trapezoid(image.shape)
    edges_roi = cv2.bitwise_and(edges, mask)
    lines = cv2.HoughLinesP(edges_roi, rho=1, theta=np.pi / 180,
                            threshold=hough_threshold,
                            minLineLength=hough_min_length, maxLineGap=hough_max_gap)
    left  = average_lane(lines, 'left', image.shape)
    right = average_lane(lines, 'right', image.shape)
    overlay = image.copy()
    for lane in (left, right):
        if lane is not None:
            cv2.line(overlay, lane[:2], lane[2:], (0, 255, 255), 8)
    out = cv2.addWeighted(image, 0.7, overlay, 0.3, 0)
    return out, left, right, (time.perf_counter() - t0) * 1000

# Construir un dict {category: {image_name: overlay_bgr}}
classical_overlays = {'easy': {}, 'medium': {}, 'hard': {}}
for category in classical_overlays.keys():
    for img_path in sorted((DATASET_DIR / category).glob('*.png')):
        img = cv2.imread(str(img_path))
        if img is None: continue
        overlay, _, _, _ = detect_lanes_classical(img)
        classical_overlays[category][img_path.name] = overlay

print(f"Overlays clásico regenerados: {sum(len(v) for v in classical_overlays.values())} imágenes")

## 2. Cargar los overlays DL desde disco

In [ ]:
DL_OVERLAYS_DIR = OUT_DIR / '03_dl_overlays'

dl_overlays = {'easy': {}, 'medium': {}, 'hard': {}}
for path in sorted(DL_OVERLAYS_DIR.glob('*.png')):
    # nombre: "<category>_<original_filename>.png"
    name = path.name
    if name.startswith('easy_'):
        cat, original = 'easy', name[len('easy_'):]
    elif name.startswith('medium_'):
        cat, original = 'medium', name[len('medium_'):]
    elif name.startswith('hard_'):
        cat, original = 'hard', name[len('hard_'):]
    else:
        continue
    dl_overlays[cat][original] = cv2.imread(str(path))

print(f"Overlays DL cargados: {sum(len(v) for v in dl_overlays.values())} imágenes")

## 3. Grid visual: original | clásico | DL

2 imágenes representativas por categoría × 3 columnas. Elige imágenes con index 0 y 5 de cada categoría (no consecutivas, para más variedad).

In [ ]:
def to_rgb(bgr):
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

rows = []
for category in ['easy', 'medium', 'hard']:
    names = sorted(classical_overlays[category].keys())
    picks = [names[0], names[len(names) // 2]] if len(names) >= 2 else names[:1]
    for n in picks:
        original = cv2.imread(str(DATASET_DIR / category / n))
        cls_img  = classical_overlays[category].get(n)
        dl_      = dl_overlays[category].get(n)
        rows.append((category, n, original, cls_img, dl_))

n_rows = len(rows)
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 3.5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, 3)
for r, (cat, name, orig, cls, dl_) in enumerate(rows):
    for c, (img, title_prefix) in enumerate([
        (orig, 'Original'),
        (cls, 'Clásico (Hough)'),
        (dl_, 'DL (ONNX)'),
    ]):
        ax = axes[r, c]
        if img is None:
            ax.text(0.5, 0.5, 'no disponible', ha='center', va='center')
        else:
            ax.imshow(to_rgb(img))
        ax.set_title(f'{title_prefix} · {cat}/{name}', fontsize=9)
        ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / '04_grid_visual.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'✅ Grid guardado en {OUT_DIR / "04_grid_visual.png"}')

## 4. Tabla cuantitativa de detección

Tasa de "ambos carriles detectados" por categoría × método. Es una métrica naïve (no mide cuán bien dibujados están) pero suficiente para ver el contraste a alto nivel.

In [ ]:
rows_table = []
for cat in ['easy', 'medium', 'hard']:
    rows_table.append({
        'category': cat,
        'classical_both': classical['detection_rate'][cat]['both'],
        'classical_any':  classical['detection_rate'][cat]['any'],
        'dl_both':        dl['detection_rate'][cat]['both'],
        'dl_any':         dl['detection_rate'][cat]['any'],
        'delta_both':     dl['detection_rate'][cat]['both'] - classical['detection_rate'][cat]['both'],
    })

print(f"{'cat':<8} {'cls (both)':>12} {'cls (any)':>12} {'dl (both)':>12} {'dl (any)':>12} {'Δboth':>10}")
print('-' * 70)
for r in rows_table:
    print(f"{r['category']:<8} "
          f"{r['classical_both']:>12.1%} {r['classical_any']:>12.1%} "
          f"{r['dl_both']:>12.1%} {r['dl_any']:>12.1%} "
          f"{r['delta_both']:>+10.1%}")

### 4.1 Gráfica de barras: tasa "ambos" por categoría

In [ ]:
cats = ['easy', 'medium', 'hard']
x = np.arange(len(cats))
w = 0.38

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - w/2, [r['classical_both'] for r in rows_table], w, label='Clásico', color='#4a6fa5', edgecolor='black')
ax.bar(x + w/2, [r['dl_both']        for r in rows_table], w, label='Deep Learning', color='#DA4544', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(cats)
ax.set_ylabel('Tasa de detección "ambos carriles"')
ax.set_ylim(0, 1.05)
ax.set_title('Tasa de detección por categoría — Clásico vs DL')
ax.legend()
for rect_set in ax.containers:
    ax.bar_label(rect_set, fmt='%.1f%%', fontsize=9, padding=2,
                  labels=[f'{r.get_height():.0%}' for r in rect_set])
plt.tight_layout()
plt.savefig(OUT_DIR / '04_bar_detection_rate.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Comparativa de tiempos

Lo más importante para ADAS: **¿llegamos a tiempo real?**

In [ ]:
# Times per imagen del DL (los tenemos en el JSON)
dl_times  = [r['time_ms'] for r in dl['per_image']]
# Times del clásico: el JSON sólo guardó la media, no individuales. Re-cronometrar es rápido (<30s).
cls_times = []
for category in ['easy', 'medium', 'hard']:
    for img_path in sorted((DATASET_DIR / category).glob('*.png')):
        img = cv2.imread(str(img_path))
        if img is None: continue
        _, _, _, ms = detect_lanes_classical(img)
        cls_times.append(ms)

fig, ax = plt.subplots(figsize=(9, 4.5))
bp = ax.boxplot([cls_times, dl_times], tick_labels=['Clásico', 'DL (ONNX)'], patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], ['#4a6fa5', '#DA4544']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(33, color='green', linestyle=':', label='30 FPS (33 ms)')
ax.set_ylabel('Tiempo por imagen (ms, log)')
ax.set_yscale('log')
ax.set_title('Distribución de tiempos por método (boxplot, log Y)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(OUT_DIR / '04_box_times.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Clásico media={np.mean(cls_times):.1f} ms  p95={np.percentile(cls_times, 95):.1f} ms")
print(f"DL      media={np.mean(dl_times):.1f} ms  p95={np.percentile(dl_times, 95):.1f} ms")
print(f"Factor de lentitud DL/clásico:  ~{np.mean(dl_times) / np.mean(cls_times):.0f}×")

## 6. Síntesis ejecutiva

Esta es la tabla que entregarías a un PM o un Tier-1. Resume **qué es mejor para qué**.

In [ ]:
synthesis = {
    'criterio': [
        'Tasa detección easy',
        'Tasa detección medium',
        'Tasa detección hard',
        'Latencia media',
        'Latencia p95',
        'Llega a 30 FPS en CPU',
        'Dependencias / setup',
        'Explicabilidad / debug',
    ],
    'clásico': [
        f"{classical['detection_rate']['easy']['both']:.0%}",
        f"{classical['detection_rate']['medium']['both']:.0%}",
        f"{classical['detection_rate']['hard']['both']:.0%}",
        f"{np.mean(cls_times):.0f} ms",
        f"{np.percentile(cls_times, 95):.0f} ms",
        'SÍ' if np.mean(cls_times) <= 33 else 'NO',
        'OpenCV (mínimas)',
        'Total (cada paso visualizable)',
    ],
    'dl': [
        f"{dl['detection_rate']['easy']['both']:.0%}",
        f"{dl['detection_rate']['medium']['both']:.0%}",
        f"{dl['detection_rate']['hard']['both']:.0%}",
        f"{np.mean(dl_times):.0f} ms",
        f"{np.percentile(dl_times, 95):.0f} ms",
        'SÍ' if np.mean(dl_times) <= 33 else 'NO',
        'ONNX runtime + 25 MB pesos',
        'Limitada (caja negra)',
    ],
}

print(f"{'criterio':<28} {'clásico':>22}    {'DL':>22}")
print('-' * 80)
for crit, c, d in zip(synthesis['criterio'], synthesis['clásico'], synthesis['dl']):
    print(f'{crit:<28} {c:>22}    {d:>22}')

## 7. Guardar el resumen final

Esto es lo que el alumno menciona en su entregable y lo que el profesor compara entre alumnos para ver consistencia.

In [ ]:
final = {
    'classical': {
        'mean_time_ms': float(np.mean(cls_times)),
        'p95_time_ms':  float(np.percentile(cls_times, 95)),
        'detection_rate': classical['detection_rate'],
    },
    'dl': {
        'mean_time_ms': float(np.mean(dl_times)),
        'p95_time_ms':  float(np.percentile(dl_times, 95)),
        'detection_rate': dl['detection_rate'],
    },
    'comparison': {
        'speedup_classical_over_dl': float(np.mean(dl_times) / np.mean(cls_times)),
        'detection_uplift_dl_over_classical_hard': (
            dl['detection_rate']['hard']['both'] - classical['detection_rate']['hard']['both']
        ),
    },
}

with open(OUT_DIR / '04_comparison_summary.json', 'w') as f:
    json.dump(final, f, indent=2)

print(f'✅ {OUT_DIR / "04_comparison_summary.json"}')
print(json.dumps(final, indent=2))

## 8. Cierre

Tienes:
- `outputs/04_grid_visual.png` — comparación visual
- `outputs/04_bar_detection_rate.png` — barras por categoría
- `outputs/04_box_times.png` — tiempos
- `outputs/04_comparison_summary.json` — métricas

Ya puedes responder las 5 preguntas guiadas en `05_preguntas.md` usando esta evidencia.

**Tip para el entregable**: las 3 imágenes PNG anteriores son las que deberían ir en el PDF que entregas.